# MICRO-TRACE: Phase P1 (C1: Future FC Prediction) Model Training & Benchmark Pipeline (`T-003`)

This reproducible notebook executes the **Phase P1 Model Training and Multi-Arm Benchmark Architecture** (`ADR-001`, `ADR-009`, PRD Section 4.6) on Google Colab with active **Tesla T4 GPU** acceleration:
1. **Environment & GPU Setup**: Connect to Google Drive, verify CUDA capability and Tesla T4 GPU device acceleration.
2. **Configuration & Data Ingestion**: Load `c1_config.yaml`, the consecutive forecasting cohort ($N=445$: Dev $N=371$, Test $N=74$), multi-omics relative abundances, and the causal visit trajectory store.
3. **Baselines Evaluation**: Establish rigorous performance bounds with **$B_0$ (Training Mean)** and **$B_1$ / $M_0$ (Most Recent FC Persistence)**.
4. **Candidate Model Benchmark**: Cross-validate and test candidate families across modeling arms $M_0$ through $M_5$, $B_3$, and $B_4$ under strict **Train-Only Fitting Invariance (`PR-03`)**:
   - **Tabular ML**: `ElasticNet` ($L_1+L_2$ CV-tuned), `RandomForest`, and `HistGradientBoosting` (native missingness handling).
   - **Deep Architecture**: `MicroTrace-Attn` (Continuous-Time Residual Cross-Attention with Fourier time-decay embeddings $\phi(\Delta t)$ and non-negotiable linear residual skip connection from anchor FC).
5. **Artifacts Generation**: Export `c1_model_benchmark_results.csv` and `c1_model_summary.json` to `micro_trace/reports/c1/`.
6. **Full Test Suite Verification**: Execute all 36 pytest unit and leakage tests directly inside the live environment.

In [1]:
import os
import sys
from pathlib import Path
import torch

# 1. Mount Google Drive if in Google Colab
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    print('Google Drive mounted successfully.')
except ImportError:
    print('Running locally; Google Drive mount not required.')

# 2. Configure sys.path for micro_trace package
colab_pkg_root = Path('/content/drive/MyDrive/Micro-Trace/micro_trace/src')
local_pkg_root = Path('.').resolve() / 'micro_trace' / 'src'

if colab_pkg_root.exists():
    if str(colab_pkg_root) not in sys.path:
        sys.path.insert(0, str(colab_pkg_root))
    print(f'Using Colab package path: {colab_pkg_root}')
elif local_pkg_root.exists():
    if str(local_pkg_root) not in sys.path:
        sys.path.insert(0, str(local_pkg_root))
    print(f'Using local package path: {local_pkg_root}')

# 3. Verify CUDA Device & T4 GPU
cuda_avail = torch.cuda.is_available()
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {cuda_avail}')
if cuda_avail:
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

# 4. Verify micro_trace import
import micro_trace
import micro_trace.c1 as c1
print(f'Imported micro_trace from: {micro_trace.__file__}')
print('Registered c1 components:', [attr for attr in dir(c1) if not attr.startswith('_')])

Google Drive mounted successfully.
Using Colab package path: /content/drive/MyDrive/Micro-Trace/micro_trace/src
PyTorch Version: 2.14.1+cu130
CUDA Available: True
GPU Device: Tesla T4
VRAM Total: 15.64 GB
Imported micro_trace from: /content/drive/MyDrive/Micro-Trace/micro_trace/src/micro_trace/__init__.py
Registered c1 components: ['TYPE_CHECKING']


## 1. Configuration & Cohort Path Resolution
Load experiment configuration `c1_config.yaml` and resolve paths for raw data, processed multi-omics, and cohort tables.

In [2]:
import yaml
import pandas as pd
from micro_trace.c1.cohort import resolve_existing_path
from micro_trace.c1.features import load_and_align_omics_tables, build_visit_trajectory_store, DEFAULT_ZERO_SUM_SAMPLES

candidate_configs = [
    Path('/content/drive/MyDrive/Micro-Trace/micro_trace/configs/c1_config.yaml'),
    Path('configs/c1_config.yaml'),
    Path('micro_trace/configs/c1_config.yaml'),
]
cfg_path = None
for p in candidate_configs:
    if p.exists():
        cfg_path = p
        break
assert cfg_path is not None, 'Configuration file c1_config.yaml not found!'

with open(cfg_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

reports_dir = Path(cfg['paths']['c1_reports_dir'])
if not reports_dir.exists():
    reports_dir = resolve_existing_path(reports_dir, [
        Path('/content/drive/MyDrive/Micro-Trace/micro_trace/reports/c1'),
        Path('micro_trace/reports/c1'),
    ])

cohort_csv = reports_dir / 'c1_cohort_consecutive_0d.csv'
pairs_df = pd.read_csv(cohort_csv)
dev_pairs = pairs_df[pairs_df['partition'] == 'dev'].copy()
test_pairs = pairs_df[pairs_df['partition'] == 'test'].copy()

# Load Multi-Omics tables
omics_dict = load_and_align_omics_tables(cfg_path)

# Load Metadata & Build Trajectory Store
raw_path = Path(cfg['paths'].get('raw_data_dir', 'micro-trace-data/raw'))
meta_file = raw_path / 'metadata' / 'hmp2_metadata_2018-08-20.csv'
if not meta_file.exists():
    meta_file = resolve_existing_path(
        meta_file,
        [
            Path('/content/drive/MyDrive/Micro-Trace/micro-trace-data/raw/metadata/hmp2_metadata_2018-08-20.csv'),
            Path('micro-trace-data/raw/metadata/hmp2_metadata_2018-08-20.csv'),
        ],
    )
meta_df = pd.read_csv(meta_file, low_memory=False)

schema_file = Path(cfg['paths'].get('schema_mapping_path', 'micro_trace/configs/schema_mapping.yaml'))
if not schema_file.exists():
    schema_file = resolve_existing_path(
        schema_file,
        [
            Path('/content/drive/MyDrive/Micro-Trace/micro_trace/configs/schema_mapping.yaml'),
            Path('micro_trace/configs/schema_mapping.yaml'),
            Path('configs/schema_mapping.yaml'),
        ],
    )
with open(schema_file, 'r', encoding='utf-8') as f:
    schema_cfg = yaml.safe_load(f)

excluded_zero = cfg.get('excluded_zero_sum_samples', DEFAULT_ZERO_SUM_SAMPLES)
visit_store_df = build_visit_trajectory_store(
    meta_df=meta_df,
    omics_dict=omics_dict,
    schema_cfg=schema_cfg,
    excluded_zero_sum_samples=excluded_zero,
)

print(f'Configuration: {cfg_path}')
print(f'Total Consecutive Pairs: {len(pairs_df)} ({len(dev_pairs)} dev, {len(test_pairs)} test [UNTOUCHED])')
print(f'Dev Target Variable: future_log10_fc_plus_1 (mean={dev_pairs["future_log10_fc_plus_1"].mean():.3f}, std={dev_pairs["future_log10_fc_plus_1"].std():.3f})')
print(f'Multi-Omics Datasets: {list(omics_dict.keys())}')
print(f'Visit Trajectory Store: {visit_store_df.shape[0]} events, {visit_store_df.shape[1]} columns')

Configuration: /content/drive/MyDrive/Micro-Trace/micro_trace/configs/c1_config.yaml
Total Consecutive Pairs: 445 (371 dev, 74 test [UNTOUCHED])
Dev Target Variable: future_log10_fc_plus_1 (mean=1.800, std=0.537)
Multi-Omics Datasets: ['species_relab', 'genus_relab', 'pathways_unstrat_relab']
Visit Trajectory Store: 1769 events, 152 columns


## 2. Baseline Models Evaluation ($B_0$ & $B_1$ / $M_0$)
Establish empirical benchmark performance anchors on the development cohort under 5-fold participant-grouped CV:
- **$B_0$ (Mean Baseline)**: Predicts constant training-set mean $\log_{10}(\text{FC}+1)$.
- **$B_1$ / $M_0$ (Persistence Baseline)**: Predicts current index $\log_{10}(\text{FC}+1)$ directly as future estimate via `C1FeaturePipeline(arm='M0')`.
- *In accordance with protocol PR-03 and PRD line 195, the 20% test partition remains 100% untouched.*

In [3]:
import numpy as np
from micro_trace.c1.features import C1FeaturePipeline
from micro_trace.c1.models import MeanBaseline, PersistenceBaseline, compute_regression_metrics

y_dev = dev_pairs['future_log10_fc_plus_1'].to_numpy(dtype=np.float64)

# B0: Mean Baseline across 5-fold CV
b0_fold_maes = []
for f in range(5):
    tr_mask = (dev_pairs['cv_fold'] != f).to_numpy()
    val_mask = (dev_pairs['cv_fold'] == f).to_numpy()
    b0 = MeanBaseline().fit(np.zeros((tr_mask.sum(), 1)), y_dev[tr_mask])
    preds_val = b0.predict(np.zeros((val_mask.sum(), 1)))
    b0_fold_maes.append(compute_regression_metrics(y_dev[val_mask], preds_val)['mae'])

# B1 / M0: Persistence Baseline via Feature Pipeline M0 across 5-fold CV
m0_fold_maes = []
for f in range(5):
    tr_mask = (dev_pairs['cv_fold'] != f).to_numpy()
    val_mask = (dev_pairs['cv_fold'] == f).to_numpy()
    pipe_m0 = C1FeaturePipeline(arm='M0')
    pipe_m0.fit(dev_pairs[tr_mask], visit_store_df, omics_dict)
    X_val_m0, _ = pipe_m0.transform(dev_pairs[val_mask], visit_store_df, omics_dict)
    pers_m0 = PersistenceBaseline(fc_col_idx=0).fit(X_val_m0, y_dev[val_mask])
    preds_val_m0 = pers_m0.predict(X_val_m0)
    m0_fold_maes.append(compute_regression_metrics(y_dev[val_mask], preds_val_m0)['mae'])

print(f'=== BASELINE EVALUATION (5-Fold Participant-Grouped CV) ===')
print(f'B0 Mean Baseline:        CV MAE = {np.mean(b0_fold_maes):.4f} (+/- {np.std(b0_fold_maes):.4f})')
print(f'B1 Persistence (M0):     CV MAE = {np.mean(m0_fold_maes):.4f} (+/- {np.std(m0_fold_maes):.4f})')

=== BASELINE EVALUATION (5-Fold Participant-Grouped CV) ===
B0 Mean Baseline:        CV MAE = 0.5007 (+/- 0.0287)
B1 Persistence (M0):     CV MAE = 0.2669 (+/- 0.0317)


## 3. Candidate Model Family Benchmark Across Modeling Arms
Execute candidate model benchmark (`ADR-001`, `ADR-009`) across all experimental arms:
- **Arms**: `M0` (Persistence), `M1` (Taxa only), `M2` (Taxa + Pathways), `M3` (Omics + Demographics), `M4` (Full Multi-Omics + Clinical + Medication), `M5` (Longitudinal Trajectory Vectors), `B3` (Clinical + Demographics), `B4` (Clinical + Demographics + Trajectory).
- **Families**: `ElasticNet`, `RandomForest`, `HistGradientBoosting`, and `MicroTrace-Attn`.
- **Validation**: Strict 5-fold participant-grouped cross-validation ($N=371$, 5 folds of 15 disjoint participants) under train-only fitting (`PR-03`).
- **Test Set Protection**: The 20% test partition ($N=74$, 18 participants) remains untouched (`evaluate_test=False`).

In [4]:
import time
from micro_trace.c1.models import run_c1_model_benchmark

t0 = time.time()
print('Starting Phase P1 Model Benchmark on Colab...')

benchmark_summary = run_c1_model_benchmark(
    cfg_path=cfg_path,
    arms=('M0', 'M1', 'M2', 'M3', 'M4', 'M5', 'B3', 'B4'),
    families=('ElasticNet', 'RandomForest', 'HistGradientBoosting', 'MicroTrace-Attn'),
    epochs_attn=25,
    evaluate_test=False,
    dev_pairs=dev_pairs,
    visit_store_df=visit_store_df,
    omics_dict=omics_dict,
)

elapsed = time.time() - t0
print(f'\nBenchmark execution completed in {elapsed:.1f} seconds!')
print(f'Best CV Model: {benchmark_summary["best_cv_model"]["arm"]} - {benchmark_summary["best_cv_model"]["model_family"]} (CV MAE: {benchmark_summary["best_cv_model"]["cv_mae_mean"]:.4f})')
print(f'Test Partition Status: {benchmark_summary["test_partition_status"]}')

Starting Phase P1 Model Benchmark on Colab...
[06:36:52] Processing Modeling Arm: M0


[06:36:52] Processing Modeling Arm: M1


  -> Arm M1 | ElasticNet: 5-fold CV MAE = 0.3116 (+/- 0.0306)


  -> Arm M1 | RandomForest: 5-fold CV MAE = 0.3230 (+/- 0.0342)


  -> Arm M1 | HistGradientBoosting: 5-fold CV MAE = 0.3359 (+/- 0.0391)


  -> Arm M1 | MicroTrace-Attn: 5-fold CV MAE = 0.2616 (+/- 0.0340)


... (364 outputs left out)

  -> Arm B4 | RandomForest: 5-fold CV MAE = 0.4765 (+/- 0.0141)


  -> Arm B4 | HistGradientBoosting: 5-fold CV MAE = 0.4875 (+/- 0.0288)


  -> Arm B4 | MicroTrace-Attn: 5-fold CV MAE = 0.2641 (+/- 0.0327)



Benchmark execution completed in 553.8 seconds!
Best CV Model: M1 - MicroTrace-Attn (CV MAE: 0.2616)
Test Partition Status: untouched_held_out


## 4. Comprehensive Benchmark Table & Artifacts Verification
Inspect the complete cross-validation performance table and compare candidates against persistence baseline $M_0$.

In [5]:
import pandas as pd

results_csv = reports_dir / 'c1_model_benchmark_results.csv'
assert results_csv.exists(), f'Results CSV not found at {results_csv}'
results_df = pd.read_csv(results_csv)

print(f'Total Evaluated Model-Arm Configurations: {len(results_df)}')
print('\n' + '='*80)
print('PHASE P1 C1 MODEL BENCHMARK RESULTS (Sorted by 5-Fold CV MAE):')
print('='*80)
display_cols = ['arm', 'model_family', 'feature_dim', 'cv_mae_mean', 'cv_mae_std', 'cv_rmse_mean', 'cv_r2_mean', 'cv_spearman_rho_mean']
sorted_df = results_df[display_cols].sort_values('cv_mae_mean')
print(sorted_df.to_string(index=False))

# Compare Top Models against Persistence (M0)
m0_cv_mae = float(results_df[results_df['arm'] == 'M0']['cv_mae_mean'].iloc[0])
best_row = sorted_df.iloc[0]
best_mae = float(best_row['cv_mae_mean'])
pct_delta = ((best_mae - m0_cv_mae) / m0_cv_mae) * 100
print(f'\nPersistence Baseline M0 CV MAE: {m0_cv_mae:.4f}')
print(f'Top Performer ({best_row["arm"]} - {best_row["model_family"]}) CV MAE: {best_mae:.4f} ({pct_delta:+.2f}% vs persistence)')

Total Evaluated Model-Arm Configurations: 30

PHASE P1 C1 MODEL BENCHMARK RESULTS (Sorted by 5-Fold CV MAE):
arm         model_family  feature_dim  cv_mae_mean  cv_mae_std  cv_rmse_mean  cv_r2_mean  cv_spearman_rho_mean
 M1      MicroTrace-Attn           29       0.2616      0.0340        0.4140      0.3878                0.6863
 M5      MicroTrace-Attn          646       0.2635      0.0324        0.4144      0.3879                0.6870
 B3      MicroTrace-Attn          130       0.2640      0.0329        0.4152      0.3850                0.6892
 B4      MicroTrace-Attn          368       0.2641      0.0327        0.4159      0.3832                0.6886
 M4      MicroTrace-Attn          524       0.2642      0.0323        0.4158      0.3837                0.6850
 M2      MicroTrace-Attn          158       0.2643      0.0326        0.4172      0.3788                0.6868
 M3      MicroTrace-Attn          396       0.2645      0.0331        0.4164      0.3814                0.6873
 M0

## 5. Full Package Test Suite Execution (`pytest`)
Verify all 36 unit, integration, and leakage prevention tests pass cleanly.

In [6]:
import pytest

test_dir = Path('/content/drive/MyDrive/Micro-Trace/micro_trace/tests')
if not test_dir.exists():
    test_dir = Path('micro_trace/tests').resolve()

print(f'Running full test suite from: {test_dir}')
exit_code = pytest.main(['-v', str(test_dir)])
print(f'\nPytest Exit Code: {exit_code}')
assert exit_code == 0, f'Test suite failed with exit code {exit_code}'
print('ALL 36 UNIT & LEAKAGE TESTS PASSED SUCCESSFULLY IN LIVE RUNTIME!')

Running full test suite from: /content/drive/MyDrive/Micro-Trace/micro_trace/tests


============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.14.17, anyio-4.16.0, platformdirs-4.14.3, typeguard-4.6.0
collecting ... 

collected 37 items

../../drive/MyDrive/Micro-Trace/micro_trace/tests/test_c1_cohort.py::test_c1_consecutive_cohort_counts_and_causality 

PASSED [  2%]


../../drive/MyDrive/Micro-Trace/micro_trace/tests/test_c1_cohort.py::test_c1_all_horizons_counts 

PASSED [  5%]

... (68 outputs left out)


../../drive/MyDrive/Micro-Trace/micro_trace/tests/test_reports.py::test_report_metrics_match_source_tables 

PASSED [100%]



============================== 37 passed in 2.96s ==============================



Pytest Exit Code: 0
ALL 36 UNIT & LEAKAGE TESTS PASSED SUCCESSFULLY IN LIVE RUNTIME!
